# Fact-checking des erreurs produites par les LLM

Ce notebook est consacré à l’analyse des réponses incorrectes produites par les modèles Gemini et OpenAI lors de l’évaluation sur les questions médicales francophones du jeu de données MediQAI. L’objectif est de dépasser la simple comparaison entre la réponse prédite et la réponse de référence afin d’identifier les raisons pour lesquelles les modèles se sont trompés.

L’analyse porte notamment sur la justification générée par chaque modèle. Celle-ci est confrontée au contexte de la question, à la réponse attendue et à des sources médicales fiables. Cette démarche permet d’évaluer la validité factuelle du raisonnement présenté et de distinguer plusieurs causes possibles d’erreur, telles qu’une affirmation médicale incorrecte, un raisonnement incohérent, une mauvaise interprétation de la question ou une information insuffisamment étayée.

Une attention particulière est également portée aux erreurs associées à un niveau de confiance élevé, car elles représentent les situations dans lesquelles un modèle produit une réponse incorrecte tout en paraissant certain de son raisonnement.


In [1]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv

In [2]:
PROJECT_ROOT = Path.cwd().parent

RESULTS_DIR = (
    PROJECT_ROOT
    / "final_evaluation"
)

RESULTS_DATA = (
    RESULTS_DIR
    / "erreur_LLM.parquet"
)

df_errors_results = pd.read_parquet(RESULTS_DATA)

In [3]:
display(df_errors_results)

,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_23245,Orthopedics,"Cas clinique :\nUn jeune homme de 25 ans, vict...",E,B,False,L'attitude vicieuse du membre inférieur droit ...,95.0
1,Gemini,mcqu_train_11747,Infectious Diseases,Cas clinique :\nVous êtes amené à voir à votre...,B,A,False,Devant une angine streptococcique (à streptoco...,95.0
2,Gemini,mcqu_train_7689,Nephro-Urology,Question :\nUne insuffisance rénale aiguë par ...,A,B,False,L'insuffisance rénale aiguë par obstacle est p...,100.0
3,Gemini,mcqu_train_18793,Pulmonology,Cas clinique :\nUn homme de 40 ans a ressenti ...,B,C,False,Chez un patient de 40 ans (ou de plus de 40 an...,90.0
4,Gemini,mcqu_train_12851,Pulmonology,"Cas clinique :\nMonsieur M., manoeuvre de son ...",C,A,False,L'image décrit une opacité en bande d'environ ...,90.0
...,...,...,...,...,...,...,...,...,...
808,OpenAI,mcqu_train_948,Cardiology,Question :\nQuelle est la dose de xylocaïne à ...,C,B,False,La xylocaïne (lidocaïne) utilisée en préventio...,88.0
809,OpenAI,mcqu_train_25805,Gynecology and Obstetrics,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc...",C,B,False,"Après un accouchement prématuré, on réalise sy...",93.0
810,OpenAI,mcqu_train_24849,Microbiology,"Question :\nEn cas de septicémie, l'antibiothé...",B,D,False,"En cas de septicémie, l’antibiothérapie doit ê...",78.0
811,OpenAI,mcqu_train_21161,Ophthalmology,Cas clinique :\nUn homme âgé de 65 ans consult...,B,A,False,Une occlusion de la veine centrale de la rétin...,78.0


Nous allons sélectionner **27 `sample_id` distincts**, chacun étant associé à **deux observations**, correspondant respectivement aux modèles **Gemini** et **OpenAI**, soit un total de **60 lignes**. **27** correspond au nombre de spécialités différentes.

La sélection est réalisée selon les critères suivants :

- le même `sample_id` doit être présent pour les deux modèles ;
- la **confiance déclarée doit être supérieure à 90 %** pour chacun des deux modèles ;
- les `medical_subject` sont, dans la mesure du possible, **diversifiés** afin d’obtenir un échantillon couvrant plusieurs domaines médicaux.

La méthode suivante permet d’effectuer cette sélection :

In [ ]:
# Garde uniquement les réponses avec une confiance > 90
df_high_confidence = df_errors_results.loc[
    df_errors_results["confidence"] > 90
].copy()

# Garde uniquement les sample_id présents pour les deux modèles
sample_model_count = (
    df_high_confidence
    .groupby("sample_id")["model_name"]
    .nunique()
)

common_sample_ids = sample_model_count[
    sample_model_count == 2
].index

df_candidates = df_high_confidence.loc[
    df_high_confidence["sample_id"].isin(common_sample_ids)
].copy()

# Sélectionne 30 sample_id en diversifiant les domaines médicaux
selected_samples = (
    df_candidates[
        ["sample_id", "medical_subject"]
    ]
    .drop_duplicates(subset="sample_id")
    .sort_values("medical_subject")
    .groupby("medical_subject")
    .head(1)
)

# Garde les 30 sample_id sélectionnés
selected_sample_ids = (
    selected_samples["sample_id"]
    .head(30)
)

# DataFrame final : 30 questions × 2 modèles = 60 lignes
df_selected = (
    df_candidates.loc[
        df_candidates["sample_id"].isin(
            selected_sample_ids
        )
    ]
    .sort_values(
        ["sample_id", "model_name"]
    )
    .reset_index(drop=True)
)

In [38]:
display(df_selected.head())

,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_10195,Neurology,"Cas clinique :\nUn homme de 57 ans, ouvrier ag...",E,A,False,Le tableau clinique évoque fortement un accide...,100.0
1,OpenAI,mcqu_train_10195,Neurology,"Cas clinique :\nUn homme de 57 ans, ouvrier ag...",E,A,False,Le tableau évoque un accident vasculaire céréb...,98.0
2,Gemini,mcqu_train_14528,Hepato-Gastroenterology,"Question :\nUn ictère progressif avec prurit, ...",B,D,False,"L'association d'un ictère nu (progressif, sans...",100.0
3,OpenAI,mcqu_train_14528,Hepato-Gastroenterology,"Question :\nUn ictère progressif avec prurit, ...",B,D,False,"Un ictère progressif, indolore, avec prurit, s...",97.0
4,Gemini,mcqu_train_19766,Occupational Medicine,"Question :\nParmi les propositions suivantes, ...",D,C,False,Le certificat médical final de guérison attest...,100.0


In [5]:
#df_selected.to_csv(RESULTS_DIR / "error_selected.csv", index=False,  encoding="utf-8-sig")

In [19]:
df_id = df_selected[["sample_id"]].copy()

In [21]:
df_id["sample_id"] = (
    df_id["sample_id"]
    .str.extract(r"(\d+)$")[0]
    .astype(int)
)

In [23]:


DATA_ROOT = (
    PROJECT_ROOT
    / "data" / "raw" / "medical"
)

DATA= (
    DATA_ROOT
    / "medical_mcqu_train.parquet"
)

df_data = pd.read_parquet(DATA)

In [ ]:
df_data["id"] = df_data["id"].astype(int)
df_filtered = df_data.loc[
    df_data["id"].isin(
        df_id["sample_id"]
    )
].copy()

In [37]:
display(df_filtered)

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type,question_length_chars,question_length_words
185,21179,"Ce nourrisson de 2 mois, vivant dans des condi...",La soeur jumelle du nourrisson élevée par la g...,Il n'y a rien à faire,Vous isolez et séparez le frère et la soeur,Vous vaccinez la soeur,Vous prescrivez un macrolide à la soeur,Vous prescrivez un aminoside à la soeur,B,QCU,Infectious Diseases,Reasoning,208,32
455,23363,None,(cochez la réponse juste) Parmi les propositio...,1+2+3,1+2+3+4,1+3+5,2+3+5,1+3+4+5,D,QCU,Rheumatology,Understanding,316,37
1392,19916,None,Quel est le médicament dont il faudra contrôle...,Antivitamine K,Héparine,Diurétique thiazidique,Lithium,Sulfamide hypoglycémiant,A,QCU,Pharmacology,Understanding,134,20
1617,26052,None,(cochez la réponse fausse) La contraception pa...,Est indiquée dans le post partum immédiat,Est indiquée chez la femme cardiaque,Est indiquée chez la femme hypertendue,Est prise de façon discontinue 21 jours par mois,Assure une contraception par effets périphériques,C,QCU,Gynecology and Obstetrics,Understanding,76,10
1817,22488,"Un patient âgé de 45 ans, consulte pour céphal...",Le malade est mis sous traitement substitutif ...,Rhinorrhée,Diabète insipide,Sinusite maxillaire,Méningite,Apoplexie hypophysaire 12.\r\n3 mois après la ...,C,QCU,Endocrinology and Metabolism,Understanding,292,44
2945,24880,None,(cochez la réponse juste) Au niveau d'une caps...,Sont groupées en capsomères.,Forment une figure géométrique bien définie.,Sont de nature glyco-protéique.,Sont antigéniques.,Les propositions A + B.,E,QCU,Microbiology,Understanding,100,16
2968,10195,"Un homme de 57 ans, ouvrier agricole, constate...",Quel examen demanderez-vous en priorité ?,Tomodensitométrie craniocérébrale,Artériographie numérisée par voie veineuse,Exploration par ultrasons des vaisseaux du cou,Electroencéphalogramme,Echocardiogramme,E,QCU,Neurology,Reasoning,41,6
2991,2096,Une femme de 62 ans est hospitalisée pour bila...,Quel est le meilleur traitement de l'intertrig...,Equilibrer le diabète sans autre mesure complé...,Application locale d'un anti-candidosique,Application locale d'un anti-candidosique et p...,Application locale d'un anti-candidosique et p...,"Prise de nystatine per os, sans traitement local",C,QCU,Dermatology,Reasoning,67,11
3392,6427,None,A partir de quel taux d'alcoolémie la conduite...,"0,60 gramme par litre","0,80\t""""","1,20\t""""","1,80\t""""","0,50\t""""",B,QCU,Forensic Medicine and Toxicology,Understanding,91,13
3443,7388,None,Lequel des tableaux suivants vous paraît évoqu...,Alopécie hippocratique,Pelade,Squames épaisses traversées par le cheveu,Alopécie de l'enfant avec squames,Alopécie diffuse de l'adulte,C,QCU,Parasitology,Understanding,97,14
